# 00 · El corpus

**Duración estimada:** 30 minutos

## Propósito

En el teórico trabajamos con cuatro oraciones inventadas. Acá empieza el corpus
real: los textos que Hernán Casciari publicó en su blog entre 2004 y 2015.

Pero antes de contar una sola palabra hay que hacer algo que en la unidad anterior
quedó dicho como principio y ahora se vuelve trabajo concreto:

> **Un corpus es texto más procedencia.** Sin eso no es un corpus, es una carpeta
> con archivos.

Los archivos que vamos a recibir no tienen procedencia. Este cuaderno se la
construye, y de paso descubre varias cosas que nadie nos avisó.

## Qué consume

- `datos/blog/` — doce archivos, uno por año

## Qué produce

- `datos/corpus.jsonl` — el corpus con procedencia
- `datos/FUENTE.md` — la documentación, escrita por ustedes

## 1. Lo que nos dieron

Doce archivos con extensión `.txt`, uno por año. Empecemos como empezaría
cualquiera: abriendo uno.

In [ ]:
# pathlib: para armar rutas que funcionen en cualquier sistema operativo.
from pathlib import Path

carpeta = Path("datos/blog")

archivos = sorted(carpeta.glob("*.txt"))

for archivo in archivos:
    tamano = archivo.stat().st_size
    print(archivo.name, "->", f"{tamano:,}", "bytes")

Ahora abrimos uno como se abre un archivo de texto.

In [ ]:
archivo = open("datos/blog/2004.txt", encoding="utf-8")

try:
    contenido = archivo.read()
    print(contenido[:200])
except Exception as error:
    print("No se pudo leer como texto.")
    print(type(error).__name__)
    print(error)

archivo.close()

**Falla.**

El mensaje dice que el byte `0x80` de la posición 0 no es un carácter UTF-8
válido. Miremos esos bytes directamente, sin intentar interpretarlos.

In [ ]:
# El modo "rb" abre el archivo en binario: devuelve los bytes crudos.
archivo = open("datos/blog/2004.txt", "rb")
primeros_bytes = archivo.read(16)
archivo.close()

print(primeros_bytes)

`\x80\x03X` es la firma de un objeto **pickle**: el formato en que Python
guarda objetos en disco.

O sea que estos archivos **no son texto**, aunque se llamen `.txt`. Son objetos de
Python serializados. Quien los guardó usó `pickle` y les puso la extensión
equivocada.

Se leen con la librería correspondiente.

In [ ]:
# pickle: para leer objetos de Python guardados en disco.
import pickle

archivo = open("datos/blog/2004.txt", "rb")
texto_2004 = pickle.load(archivo)
archivo.close()

print("tipo:", type(texto_2004).__name__)
print("caracteres:", f"{len(texto_2004):,}")
print()
print(texto_2004[:300])

> **Una advertencia que conviene tener presente.** Abrir un pickle **ejecuta
> código**. Un archivo pickle malicioso puede hacer cualquier cosa en tu máquina
> al abrirlo.
>
> Nunca abras un pickle que no armaste vos o que no venga de una fuente en la que
> confiás. Para compartir datos existen formatos que no ejecutan nada: `.txt`,
> `.csv`, `.json`, `.jsonl`.
>
> Es una de las razones para convertir este corpus.

## 2. Qué hay adentro

Ya podemos leerlo. Miremos qué forma tiene.

In [ ]:
textos_por_anio = {}

for archivo_del_anio in archivos:
    anio = int(archivo_del_anio.stem)
    archivo = open(archivo_del_anio, "rb")
    textos_por_anio[anio] = pickle.load(archivo)
    archivo.close()

print("años cargados:", sorted(textos_por_anio.keys()))
print()

total = 0
for anio in sorted(textos_por_anio):
    largo = len(textos_por_anio[anio])
    total = total + largo
    print(anio, "->", f"{largo:,}", "caracteres")

print()
print("total:", f"{total:,}", "caracteres")

Dos millones y medio de caracteres. Es un corpus de buen tamaño.

Ahora la pregunta importante: **¿dónde termina un texto y empieza el siguiente?**

In [ ]:
saltos_de_linea = 0
for anio in textos_por_anio:
    saltos_de_linea = saltos_de_linea + textos_por_anio[anio].count("\n")

print("saltos de línea en todo el corpus:", saltos_de_linea)
print("caracteres por cada salto de línea:", f"{total // saltos_de_linea:,}")

**115 saltos de línea en 2.376.248 caracteres.**

No hay párrafos. No hay títulos. No hay fechas. Cada archivo es **un solo string
gigante** con todos los textos de ese año pegados uno detrás de otro.

El scrapeo original bajó los textos por separado y los concatenó por año antes de
guardarlos. Esa información ya no está y no se puede recuperar: el sitio original
ya no existe.

> **Consecuencia práctica.** Con este corpus **no se puede preguntar nada sobre un
> texto individual**: no se puede comparar un cuento con otro, ni contar en
> cuántos cuentos aparece una palabra. Si tu pregunta necesita eso, este corpus no
> sirve.
>
> Eso no lo dice ningún archivo. Lo averiguamos mirando.

## 3. Lo que sí se puede recuperar

Miremos un tramo del texto con atención.

In [ ]:
print(texto_2004[380:520])

Fijate en `lector.La idea nació`.

Un punto pegado a una mayúscula, **sin espacio en el medio**. Eso no pasa cuando
uno escribe: pasa cuando se pegan dos párrafos y se borra el salto de línea.

Es una huella. Y se puede contar.

In [ ]:
# re: expresiones regulares, para buscar patrones en el texto.
import re

# Un punto seguido de mayúscula SIN espacio: la huella del párrafo perdido.
patron_pegado = r'[a-záéíóúñ]\.[A-ZÁÉÍÓÚÑ]'

# Un punto seguido de mayúscula CON espacio: un final de oración normal.
patron_normal = r'[a-záéíóúñ]\. [A-ZÁÉÍÓÚÑ]'

pegados = 0
normales = 0
for anio in textos_por_anio:
    pegados = pegados + len(re.findall(patron_pegado, textos_por_anio[anio]))
    normales = normales + len(re.findall(patron_normal, textos_por_anio[anio]))

print("puntos pegados a la mayúscula:", f"{pegados:,}")
print("puntos con espacio           :", f"{normales:,}")
print()
porcentaje = 100 * pegados / (pegados + normales)
print("porcentaje de fronteras rotas:", f"{porcentaje:.0f}%")

**Una de cada cuatro fronteras de oración está rota.**

Eso hoy no nos molesta —vamos a contar palabras, no oraciones— pero en la unidad
de spaCy sí va a importar, porque ahí el análisis se apoya en las oraciones.

Por ahora nos sirve para otra cosa: **cada punto pegado marca dónde terminaba un
párrafo**, y eso nos permite reconstruir una unidad intermedia entre "el año
entero" y "el texto individual que se perdió".

## 4. Construir el corpus

Vamos a cortar cada año en párrafos y guardar cada párrafo con su procedencia.

El formato es `.jsonl`: un objeto JSON por línea. Es el mismo que usaron en la
unidad de adquisición, y tiene tres virtudes: se lee con un editor de texto, no
ejecuta código, y se puede procesar línea por línea sin cargar todo en memoria.

In [ ]:
def cortar_en_parrafos(texto):
    # Corta donde hay un punto pegado a una mayúscula.
    # El punto NO se consume: por eso cada párrafo conserva su puntuación final.
    marca = r'(?<=[a-záéíóúñ0-9\)\!\?\»"])\.(?=[A-ZÁÉÍÓÚÑ¿«"])'
    partes = re.split(marca, texto)

    parrafos = []
    for parte in partes:
        limpio = parte.strip()
        if len(limpio) < 2:
            continue
        if not limpio.endswith((".", "!", "?", "…", "»", '"')):
            limpio = limpio + "."
        parrafos.append(limpio)
    return parrafos


# Probamos con un año antes de hacerlo con todos.
parrafos_2004 = cortar_en_parrafos(texto_2004)

print("párrafos en 2004:", len(parrafos_2004))
print()
print("primero:", parrafos_2004[0][:120], "...")
print()
print("segundo:", parrafos_2004[1][:120], "...")

Los cortes caen exactamente donde terminaba un párrafo. Funciona.

Ahora los doce años.

In [ ]:
registros = []

for anio in sorted(textos_por_anio):
    parrafos = cortar_en_parrafos(textos_por_anio[anio])
    numero = 0
    for parrafo in parrafos:
        numero = numero + 1
        registro = {
            "anio": anio,
            "parrafo": numero,
            "caracteres": len(parrafo),
            "texto": parrafo,
        }
        registros.append(registro)

print("párrafos recuperados:", f"{len(registros):,}")

### Verificar antes de guardar

Antes de dar por bueno el resultado, comprobemos que no perdimos texto.

In [ ]:
caracteres_recuperados = 0
for registro in registros:
    caracteres_recuperados = caracteres_recuperados + registro["caracteres"]

print("caracteres en blog/    :", f"{total:,}")
print("caracteres recuperados :", f"{caracteres_recuperados:,}")
print("conservado             :", f"{100 * caracteres_recuperados / total:.2f}%")

Los caracteres que faltan son los espacios en blanco que sacamos con `.strip()`
al principio y al final de cada párrafo.

**Verificar es parte del trabajo, no un extra.** Si esta cuenta hubiera dado 80%,
habríamos perdido una quinta parte del corpus sin enterarnos.

In [ ]:
# json: para escribir el archivo, un objeto por línea.
import json

archivo = open("datos/corpus.jsonl", "w", encoding="utf-8")

for registro in registros:
    # ensure_ascii=False para que los acentos se guarden como acentos
    # y el archivo se pueda leer con un editor de texto.
    linea = json.dumps(registro, ensure_ascii=False)
    archivo.write(linea + "\n")

archivo.close()

print("escrito: datos/corpus.jsonl")

Abrilo con un editor de texto antes de seguir. **Ahora sí se puede leer.**

In [ ]:
# pandas: para trabajarlo como tabla.
import pandas as pd

corpus = pd.read_json("datos/corpus.jsonl", lines=True)

print("filas:", f"{len(corpus):,}")
print()
corpus.head(3)

In [ ]:
parrafos_por_anio = corpus.groupby("anio").size()

print(parrafos_por_anio)

## 5. La procedencia

Falta la mitad del trabajo. Un corpus es texto **más procedencia**, y hasta ahora
solo tenemos el texto.

La celda siguiente escribe `datos/FUENTE.md`. **Léanlo y complétenlo**: hay cosas
que solo ustedes pueden decidir, como si los párrafos de menos de 40 caracteres
son ruido o son datos.

In [ ]:
documentacion = '''# FUENTE — corpus de textos de Hernán Casciari

## Procedencia

**Textos:** entradas del blog de Hernán Casciari, 2004 a 2015.

**Captura:** no la hicimos nosotros. Los archivos vienen de un ejercicio de NLP
publicado en Aprende Machine Learning, que scrapeó 386 textos del sitio de
Editorial Orsai en enero de 2019.

**Estado de la fuente:** los textos ya no están disponibles en línea. No se
pueden volver a bajar ni verificar contra el original.

**Uso:** material de trabajo para una materia. Los textos son de su autor.

## Qué recibimos

Doce archivos con extensión `.txt` que en realidad son objetos `pickle`.
Cada uno contiene un único string con todos los textos de ese año concatenados.

## Qué se perdió

- Los límites de cada texto: el scrapeo bajó 386 textos y los pegó por año.
- Los títulos.
- Las fechas de publicación. Solo sobrevive el año.
- Las URLs.

**Con este corpus no se puede preguntar nada sobre un texto individual.**

## Qué reconstruimos

Los párrafos, a partir de una huella: al concatenar se borró el salto de línea
pero quedó el punto pegado a la mayúscula siguiente.

- párrafos recuperados: COMPLETAR
- caracteres conservados: COMPLETAR

## La unidad de análisis

Cada fila de `corpus.jsonl` es un párrafo, con el año al que pertenece.
Según la pregunta, se puede trabajar por párrafo o agrupar por año.

## Decisiones pendientes

- Los párrafos de menos de 40 caracteres: ¿son ruido o son datos?
  DECIDIR Y JUSTIFICAR.
'''

archivo = open("datos/FUENTE.md", "w", encoding="utf-8")
archivo.write(documentacion)
archivo.close()

print("escrito: datos/FUENTE.md")
print()
print("Abrilo y completá lo que dice COMPLETAR y DECIDIR.")

> ### Perilla
>
> En `cortar_en_parrafos` descartamos los fragmentos de menos de 2 caracteres.
> Podríamos haber usado 40, o 100, y el corpus sería distinto.
>
> Mirá qué hay abajo de 40 caracteres y decidí vos. **No hay respuesta correcta:
> depende de qué vayas a preguntar.** Lo que no se puede es filtrarlos sin
> decirlo.

In [ ]:
cortos = corpus[corpus["caracteres"] < 40]

print("párrafos de menos de 40 caracteres:", len(cortos))
print()

for texto in cortos["texto"].head(8):
    print(" ", texto)

## Lo que hiciste hoy

- Descubriste que un archivo `.txt` puede no ser texto.
- Aprendiste por qué `pickle` no sirve para compartir datos.
- Averiguaste, mirando, qué preguntas **no** se le pueden hacer a este corpus.
- Reconstruiste los párrafos a partir de una huella del daño.
- Verificaste que no se perdiera texto.
- Escribiste la procedencia.

## Lo que viene

En el `01` empezamos a contar. Y ahí aparece la primera trampa: contar no es tan
simple como parece cuando los documentos tienen tamaños distintos.